# CMP461 Pattern Recognition - Midterm Project
## Phase 3: CNN (Convolutional Neural Network) Experimental Design

**Author:** Efe Yaşar  
**Student ID:** 210408030 

--- 

### Objective / Amaç
Testing 5 CNN architectures with varying depth and width to analyze spatial feature extraction. We will track Accuracy, Precision, Recall, and F1-Score while visualizing the convergence for each model.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix

RESULTS_DIR = "results/cnn"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs("models", exist_ok=True)

data = np.load("preprocessed_data.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_val, y_val = data["X_val"], data["y_val"]
X_test, y_test = data["X_test"], data["y_test"]

classes = ["Cats", "Dogs", "Snakes"]

In [ ]:
def build_cnn(config):
    model = models.Sequential()
    model.add(layers.Input(shape=(128, 128, 3)))
    for filters in config['layers']:
        model.add(layers.Conv2D(filters, (3, 3), activation='relu'))
        model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Flatten())
    model.add(layers.Dense(config['dense_size'], activation='relu'))
    model.add(layers.Dense(3, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

experiments = [
    {"name": "CNN_V1_Simple", "layers": [32, 64], "dense_size": 64},
    {"name": "CNN_V2_Deep", "layers": [32, 64, 128], "dense_size": 128},
    {"name": "CNN_V3_Wide", "layers": [64, 128], "dense_size": 256},
    {"name": "CNN_V4_Minimal", "layers": [16, 32], "dense_size": 32},
    {"name": "CNN_V5_Optimized", "layers": [32, 64, 128, 256], "dense_size": 256}
]

In [ ]:
results = []
for config in experiments:
    print(f"🚀 Training {config['name']}...")
    model = build_cnn(config)
    history = model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=15, batch_size=32, verbose=1)
    
    # Plots
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1); plt.plot(history.history['accuracy'], label='Train'); plt.plot(history.history['val_accuracy'], label='Val'); plt.legend(); plt.title(f"{config['name']} Acc")
    plt.subplot(1, 2, 2); plt.plot(history.history['loss'], label='Train'); plt.plot(history.history['val_loss'], label='Val'); plt.legend(); plt.title(f"{config['name']} Loss")
    plt.savefig(f"{RESULTS_DIR}/{config['name']}_history.png")
    plt.show()

    y_pred = np.argmax(model.predict(X_test), axis=1)
    p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)
    acc = np.mean(y_pred == y_test)
    
    results.append({"Model": config['name'], "Test Accuracy": acc, "Precision (Macro)": p, "Recall (Macro)": r, "F1-Score (Macro)": f1})
    
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=classes, yticklabels=classes, cmap='Greens')
    plt.title(f"{config['name']} Confusion Matrix")
    plt.savefig(f"{RESULTS_DIR}/{config['name']}_cm.png")
    plt.show()
    
    model.save(f"models/{config['name']}.h5")

df_res = pd.DataFrame(results)
df_res.to_csv(f"{RESULTS_DIR}/cnn_results.csv", index=False)
display(df_res)